# I have following code ... To my understanding my optimization code should produce similar result as PCA. Why I do get another basis?

```python
import numpy as npimport einopsimport torchimport torch.nn.functional as Ffrom sklearn.preprocessing import StandardScalerdef v2sm(v, pad_value=None): assert v.ndim == 1 root = np.sqrt(v.shape[0]) sz = int(root) if sz * sz != v.shape[0]: assert pad_value is not None, f'Failed to make square matrix without padding from vector of size {v.shape}' sz = int(np.ceil(root)) pad_elems_count = (sz * sz) - len(v) assert pad_elems_count > 0 return np.r_[v, np.full(pad_elems_count, pad_value)].reshape(sz, sz) else: return v.reshape(sz, sz)RNG = np.random.default_rng()EMBEDDING_SIZE = 9FEATURES_BANK_SIZE = 9# Стартуем с мусора и пытаемся получить хороший базис через градиентный спускfeatures = RNG.uniform(-1, +1, size=(FEATURES_BANK_SIZE, EMBEDDING_SIZE)).astype('float32')features = (features.T / np.linalg.norm(features, axis=1)).T # features axes are in rowsfeatures = torch.tensor(features, requires_grad=True)box_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))box_sample[0] = 1box_sample[-1] = 1box_sample[:,0] = 1box_sample[:,-1] = 1vert_line_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))vert_line_sample[:,1] = 1horz_line_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))horz_line_sample[1] = 1diag_line_sample = np.eye(int(np.sqrt(FEATURES_BANK_SIZE)))dataset = np.zeros((1000, *box_sample.shape), dtype='float32')samples = [box_sample, vert_line_sample, horz_line_sample, diag_line_sample]for i, sample_ind in enumerate(RNG.choice(len(samples), len(dataset), p=[0.5, 0.2, 0.2, 0.1])): dataset[i] = samples[sample_ind]dataset = einops.rearrange(dataset, 'b h w -> b (h w)')mixin_random_samples = RNG.uniform(0, 1, size=(100, FEATURES_BANK_SIZE)).astype('float32') dataset = np.r_[dataset, mixin_random_samples]scaler = StandardScaler()dataset = scaler.fit_transform(dataset)# decorrelate feature axes by gradient descent optimization procedureoptimizer = torch.optim.Adam([features], lr=0.0001)epochs_count = 30_000batch_size = 100for epoch in range(epochs_count + 1): optimizer.zero_grad() dataset_inds = RNG.choice(len(dataset), size=batch_size, replace=False) batch = dataset[dataset_inds] batch = torch.tensor(batch) # encode -> make embedding for each sample in batch: scale each feature by corresponding value, sum all features (disentagled to distributed representation) embeddings = batch @ features # decode -> extract feature values from embedding batch_hat = embeddings @ features.T loss = F.mse_loss(batch_hat, batch) if epoch > 0: loss.backward() optimizer.step() with torch.no_grad(): features[:] = (features.T / torch.linalg.vector_norm(features, dim=1)).TU, S, V = np.linalg.svd(dataset)eigvecs = V.Tfor i in range(len(eigvecs)): eigvec = eigvecs[i] dps = eigvec @ features.T.detach().cpu().numpy() print(f'eigvec #{i}, max dot prod={np.max(np.abs(dps)):.2f}')
```

Your optimization code produces a different basis than PCA because a linear autoencoder is only guaranteed to find the same principal subspace as PCA, not the same principal components (eigenvectors). [1, 2] 
While both methods minimize reconstruction error, they arrive at different solutions for the following reasons:

**1. Lack of Orthogonality and Ordering Constraints**

* PCA strictly requires that each principal component is orthogonal to the others and ordered by the amount of variance they capture.
* Linear Autoencoders have no such internal constraint. Any basis that spans the same subspace will yield the same Mean Squared Error (MSE). Because your gradient descent starts with random weights, it converges to an arbitrary (often rotated or skewed) basis within that optimal subspace. [2, 3, 4, 5, 6] 

**2. The Identity Loophole**

* In your code, you use FEATURES_BANK_SIZE = 9 and an EMBEDDING_SIZE = 9. When the bottleneck dimension equals the input dimension, the model can simply learn the identity function.
* Unlike PCA, which finds a specific ordered basis even for full-rank data, a linear autoencoder with a full-rank bottleneck has infinitely many solutions (any invertible matrix $W$ where $W^T W = I$ or even non-orthogonal pairs that satisfy reconstruction) that all achieve near-zero loss. [7, 8, 9] 

**3. Rotational Invariance of MSE**

* The reconstruction loss $||X - XWW^T||^2$ is rotationally invariant. If $W$ is an optimal solution, then $W R$ (where $R$ is a rotation matrix) is also an optimal solution.
* Gradient descent will stop as soon as it finds any such $W$ that minimizes the error, whereas PCA specifically picks the one aligned with the axes of maximum variance. [10, 11, 12] 

**How to recover PCA-like results**

If you want your autoencoder weights to match PCA eigenvectors, you can:

* Apply SVD to your weights: Perform Singular Value Decomposition on your trained features matrix. The resulting $V$ matrix will often closely match the PCA loading vectors.
* Add Regularization: Use a "tied weights" approach or add a penalty term that encourages the weights to be orthogonal and the latent dimensions to be uncorrelated. [4, 10, 11, 12] 

Would you like to see how to modify your loss function to enforce orthogonality during training?

[1] [https://arxiv.org](https://arxiv.org/pdf/1804.10253)
[2] [https://www.youtube.com](https://www.youtube.com/watch?v=hbU7nbVDzGE&t=238)
[3] [https://stats.stackexchange.com](https://stats.stackexchange.com/questions/468913/pca-vs-linear-autoencoder-features-independence)
[4] [https://arxiv.org](https://arxiv.org/abs/1904.01277)
[5] [https://www.cs.cmu.edu](https://www.cs.cmu.edu/~mgormley/courses/10601-f23//slides/lecture23-pca-ink.pdf)
[6] [https://medium.com](https://medium.com/data-science/a-deep-dive-into-autoencoders-and-their-relationship-to-pca-and-svd-97e37c81898a)
[7] [https://www.reddit.com](https://www.reddit.com/r/MachineLearning/comments/1gtng8q/d_pca_vs_autoencoders_for_dimensionality_reduction/)
[8] [https://www.youtube.com](https://www.youtube.com/watch?v=ZUbW0dsLEB4&t=96)
[9] [https://stats.stackexchange.com](https://stats.stackexchange.com/questions/468913/pca-vs-linear-autoencoder-features-independence)
[10] [https://medium.com](https://medium.com/@rasheedatsikiru/recovering-principal-components-from-linear-autoencoders-a-hands-on-guide-ae08598bbb8e)
[11] [https://proceedings.neurips.cc](https://proceedings.neurips.cc/paper/2020/file/4dd9cec1c21bc54eecb53786a2c5fa09-Paper.pdf)
[12] [https://medium.com](https://medium.com/@rasheedatsikiru/recovering-principal-components-from-linear-autoencoders-a-hands-on-guide-ae08598bbb8e)



# I added mixin_random_samples which introduces noise and I have verified that features I obtain after training are orthogonal. But they still don't align with PCA. My understanding was that prevalence of box_sample in dataset will nudge optimization to be more like PCA. But where I am wrong?

Even with orthogonality and noise, your linear autoencoder is not guaranteed to align with PCA because your objective function still lacks variance ordering and uniqueness constraints. [1, 2] 

While the [PCA algorithm](https://en.wikipedia.org/wiki/Principal_component_analysis) and your linear autoencoder both minimize reconstruction error, they differ in how they define the basis: [2, 3] 

* Principal Subspace vs. Principal Components: A linear autoencoder is only guaranteed to find the same subspace as PCA. Any [orthonormal basis](https://en.wikipedia.org/wiki/Orthogonality) within that subspace yields the same Mean Squared Error (MSE). Because you have 9 features and an embedding size of 9, there is an infinite number of orthogonal rotations that provide perfect reconstruction.
* Lack of Variance Ranking: PCA strictly orders components by the amount of variance they capture ($PC_1 > PC_2 > \dots$). Your gradient descent treats all features equally; it has no incentive to pack the most variance into "feature #0".
* Rotation Invariance: Since your loss is based on the total reconstruction error ($||X - XWW^T||^2$), the solution is rotationally invariant. Any rotation of the PCA eigenvectors is a valid global minimum for your optimizer. [2, 4] 

**How to align your code with PCA**

To force your features to align with the actual PCA eigenvectors (the "Principal Components"), you must introduce a nested objective or a bottleneck hierarchy:

   1. Iterative Training: Train for only 1 latent dimension first to find $PC_1$. Fix it, then train for the second to find $PC_2$.
   2. Weighted MSE: Scale the loss of each reconstructed dimension such that the network is forced to prioritize the first few dimensions.
   3. SVD Post-Processing: The most common way to recover PCA from an autoencoder is to simply run Singular Value Decomposition (SVD) on your trained features matrix. [2, 4] 

Would you like a code snippet showing how to add a "Variance Ranking" loss to your training loop?

[1] [https://stackoverflow.com](https://stackoverflow.com/questions/58137488/pca-features-do-not-match-original-features)
[2] [https://medium.com](https://medium.com/@rasheedatsikiru/recovering-principal-components-from-linear-autoencoders-a-hands-on-guide-ae08598bbb8e)
[3] [https://stats.stackexchange.com](https://stats.stackexchange.com/questions/288504/does-it-make-sense-to-implement-a-pca-after-an-autoencoder-processing)
[4] [https://arxiv.org](https://arxiv.org/abs/1904.01277)


# Yes, I would like

To align your autoencoder with PCA, you need to replace the standard MSE loss with a nested reconstruction loss.
Standard MSE only cares about the final reconstruction, allowing variance to be distributed randomly across your 9 features. To act like PCA, the model must be forced to capture as much variance as possible in the first feature, then the second, and so on.
The "Variance Ranking" Loss
Replace your current loss calculation with this logic. It forces the network to minimize reconstruction error at every rank level simultaneously.

** Replace your training loop's loss calculation with this:**

optimizer.zero_grad()

Standard forward passembeddings = batch @ features# batch_hat = embeddings @ features.T # No longer using this directly

**1. Start with zero cumulative lossloss = 0**

**2. Add reconstruction loss for every possible bottleneck size (1 to 9)# This forces "features[0]" to be the most important, "features[1]" second, etc.for i in range(1, EMBEDDING_SIZE + 1):**
```python
    # Extract only the first 'i' components
    partial_features = features[:i, :]
    partial_embeddings = batch @ partial_features.T
    
    # Reconstruct using ONLY these components
    partial_reconstruction = partial_embeddings @ partial_features
    
    # Add to cumulative loss
    loss += F.mse_loss(partial_reconstruction, batch)
```

**3. Backpropagate the total "ranked" loss**
```python
loss.backward()
optimizer.step()
```

**Why this works**

* Hierarchical Importance: By calculating F.mse_loss for a 1-feature bottleneck, a 2-feature bottleneck, and so on, you are telling the optimizer that the first row of your features matrix must be the [Principal Component](https://en.wikipedia.org/wiki/Principal_component_analysis) ($PC_1$) because it is the only one available to solve the first loss term.
* Breaking Rotational Invariance: Standard MSE doesn't care if your data is rotated. This nested loss breaks that symmetry by assigning a specific "rank" to each row in your basis.
* Orthogonality: Since you are already normalizing your features in your with torch.no_grad(): block, this new loss will naturally push the features toward the orthogonal eigenvectors found by Singular Value Decomposition (SVD).

**Pro-Tip for Convergence**

Because this loss is much "stricter" than your previous one, you might need to increase your learning rate (try 0.001) or increase the number of epochs to reach the same level of precision as the scikit-learn StandardScaler and PCA tools.

Would you like to see how to implement this using matrix operations instead of a loop to speed up training?



# I am confused with this snippet since features (a-ka axes) are in rows. In original code each element of a batch row scaled a corresponding feature and then all scaled features (axes, directions) were summed up (vector addition). In this snippet we deal with projection of each batch row as a WHOLE on selected number of feature rows. How doe is make sense?
```python
partial_features = features[:i, :]
partial_embeddings = batch @ partial_features.T
```

You're right to be skeptical—it looks different, but mathematically, it's the exact same operation expressed in a more efficient way.

In your original code, you used batch @ features. Because batch is $(B \times 9)$ and features is $(9 \times 9)$, each row of the output is a linear combination of the rows in features.
Specifically, each value in a batch row acts as a scalar weight for the corresponding row in features. When you do the matrix multiply, you are essentially saying:

$$Embedding = \sum_{i=1}^{9} (\text{batch value}_i \times \text{feature row}_i)$$ 

**Why the "Projection" snippet works:**

When we do partial_features = features[:i, :], we are simply discarding some of the rows (axes) you've been training.

   1. The Projection: batch @ partial_features.T calculates the dot product between your data and each of the first $i$ feature rows. This tells you "how much" of each feature is present in the data (the scalar weights).
   2. The Reconstruction: partial_embeddings @ partial_features then takes those weights and multiplies them back by the feature rows to recreate the original vector.

**How this maps to your "Sum of Scaled Features" mental model:**

If you only have 3 features ($i=3$), the reconstruction for a single sample is:

$$\text{Reconstruction} = w_1\mathbf{f}_1 + w_2\mathbf{f}_2 + w_3\mathbf{f}_3$$ ...where $w$ are the weights (embeddings) and $\mathbf{f}$ are the feature rows.

By forcing the model to minimize F.mse_loss using only the first 1 row, then only the first 2 rows, etc., you are stripping away the "Identity Loophole." The model can no longer just shuffle information across all 9 rows. To minimize the first term of the loss, row #0 MUST become the axis of maximum variance (the first Principal Component).

**Why use this instead of your original logic?**

It's just cleaner notation for the same vector addition. In your original code:

* embeddings = batch @ features (Calculates all 9 weights)
* batch_hat = embeddings @ features.T (Sums all 9 scaled features)

In the snippet, we just "cut the cord" after $i$ features to see how well the model performs with a restricted bank.

Would you like the vectorized version of this loss so you don't have to use a for loop inside your training step?


# Still didn't catch quite well. Let's consider last cycle step when i=EMBEDDING_SIZE. In this case partial_features will equal to features and hence partial_embeddings will equal to batch @ features.T. But this is not the same as batch @ features. My confusion stems from this fact!

You’ve spotted the exact point of confusion: the shape and role of your features matrix.

In your original code, you treated features as a bank of 9 basis vectors (rows). To follow your "sum of scaled features" logic, you used:

   1. embeddings = batch @ features (Weights for the features)
   2. batch_hat = embeddings @ features.T (Weighted sum of features)

**Why this was mathematically "flipped"**

In standard Linear Algebra (and PCA):

* Encoding (Projecting data onto axes): batch @ features.T
* Decoding (Reconstructing from weights): embeddings @ features

In your code, you did the opposite. Because your features matrix is square ($9 \times 9$), the math "works" both ways, but it changes which dimension represents the "feature" and which represents the "coordinate."

**The fix for your mental model**

To align with PCA and the "partial" snippet, your features matrix should be viewed as a set of 9 rows, where each row is a Principal Component.
If you use my snippet:

Encoding: How much of each 'feature row' is in the 'batch'?partial_embeddings = batch @ partial_features.T 
Decoding: Sum those 'feature rows' using the weights we just foundpartial_reconstruction = partial_embeddings @ partial_features

When i = 9, partial_features is just features. The reconstruction becomes:
batch_hat = (batch @ features.T) @ features

**Why your original code used batch @ features**

In your original loop, you were essentially treating the columns of your features matrix as the basis vectors during the encoding step, and the rows as the basis vectors during the decoding step. While this converges to a solution, it makes it impossible to "rank" them (like PCA) because the rows and columns are entangled.

By switching to the Projection/Reconstruction pattern (@ features.T then @ features), you ensure that Row 1 is the only thing that matters for the first term of the loss, Row 2 for the second, and so on.

Does it make sense now why we need to use features.T for the projection to keep the "features as rows" logic consistent?



# But why then `embeddings = batch @ features` is left in code? Is it just a leftover and it is used to update gradients?

You caught it—it is a leftover from the original logic. In the revised nested loss approach, that line is redundant and mathematically inconsistent with how PCA-like features (rows) are extracted.

In the nested loss version, you should remove embeddings = batch @ features entirely. Instead, the "embedding" (the coordinates/weights) is calculated dynamically inside the loop for each rank level.

Here is how the cleaned-up training step looks, aligning with your "features are rows" mental model:
```python
for epoch in range(epochs_count + 1):
    optimizer.zero_grad()
    
    # ... batch selection logic ...
    batch = torch.tensor(batch) 

    total_loss = 0
    # We iterate 1..9 to force row 0 to be PC1, row 1 to be PC2, etc.
    for i in range(1, FEATURES_BANK_SIZE + 1):
        # 1. Select the first 'i' rows as our basis
        sub_basis = features[:i, :] 
        
        # 2. ENCODE: Project batch onto these rows to get 'weights'
        # (Batch x 9) @ (9 x i) -> (Batch x i)
        coords = batch @ sub_basis.T 
        
        # 3. DECODE: Reconstruct by summing the scaled rows
        # (Batch x i) @ (i x 9) -> (Batch x 9)
        reconstruction = coords @ sub_basis
        
        # 4. Accumulate loss for this specific rank
        total_loss += F.mse_loss(reconstruction, batch)

    if epoch > 0:
        total_loss.backward()
        optimizer.step()
    
    # ... normalization logic ...
```

Why the original embeddings = batch @ features fails here:

   1. Dimension Mismatch: If you use your original embeddings, you have a fixed $(B \times 9)$ matrix. You can't easily "truncate" it to see how the first 2 features perform unless you are certain the first 2 columns of the embedding correspond exactly to the first 2 rows of the features.
   2. Directionality: In your original batch @ features, the columns of features were acting as the encoding filters. In PCA, the rows (eigenvectors) are both the filters (encoding) and the basis (decoding).

By using batch @ sub_basis.T inside the loop, you force the rows of your features matrix to satisfy both roles, which is exactly what PCA does.

Would you like to see how to replace this for loop with a single matrix operation to make the training much faster?



# Let's step back. In my original mental model I was used features as rows since it is easier for me to think of embedding as a linear combination of feature rows when elements of each batch rows act as coefficients. In your code features as still in rows but embedding is created as a projection of a whole batch (a whole vector) onto each of such rows (treated as principal axes). Is these two views equivalent or they distinguish in some way?

They are mathematically equivalent, but they represent two sides of the same coin: Synthesis (your view) vs. Analysis (the PCA view).

The confusion usually stems from which part of the process we focus on. Let’s break it down:

**1. Your View: Synthesis (The "Recipe")**

You think of an Embedding as a set of instructions: "Take 0.5 of Feature Row A and 1.2 of Feature Row B."

* The Math: $\text{Vector} = \sum (\text{coefficient}_i \times \text{FeatureRow}_i)$
* In Code: embedding @ features
* Role: This is the Decoder. It builds the signal from the components.

**2. The PCA View: Analysis (The "Filter")**

PCA needs to find those coefficients (the "0.5" and "1.2") by looking at the raw data. It treats the feature rows as axes and projects the data onto them.

* The Math: $\text{coefficient}_i = \text{Data} \cdot \text{FeatureRow}_i$
* In Code: batch @ features.T
* Role: This is the Encoder. It extracts the coefficients from the signal.

**Why they must be the same for PCA**

In a standard Autoencoder, the Encoder and Decoder can be totally different matrices. However, PCA is a special case where the Encoder is simply the transpose of the Decoder.
When you say:

"Embedding is created as a projection of a whole batch onto each of such rows."

You are simply describing the only way to mathematically find the specific coefficients that satisfy your mental model (the linear combination) while minimizing error.
Where the "Basis" Alignment Happens

In your original code, because you had 9 features and 9 slots, the model could pick any 9 directions that spanned the space. It was like having a recipe where you could swap "salt" for "sea salt"—the final dish (reconstruction) stays the same.

By using the nested loss (the for loop), you force the model to make the best possible "dish" using only Row 0, then the best using Rows 0 and 1. This forces Row 0 to become the "most important ingredient" (the 1st Principal Component), Row 1 the second, and so on.

Does this help bridge the gap between "building a vector from rows" and "projecting a vector onto rows"?



# But is it possible to get results like PCA when staying in Synthesis view? I mean I want to operate feature vectors as a liner combination but still get them to align with PCA in the end

Yes, you can absolutely stay in the Synthesis (linear combination) view and still align with PCA. To do this, you must change how you calculate the error during training.
In your current code, you only care about the final result (the sum of all 9 features). To get PCA, you must force the model to "build" the reconstruction one feature at a time and minimize the error at every step.

The "Step-by-Step" Synthesis Loss

Instead of summing all 9 features and checking the error once, you check the error after adding the 1st feature, then again after adding the 2nd, and so on.

# Stay in Synthesis View: batch elements are coefficients (weights)# features are your 'basis rows'
```python
optimizer.zero_grad()total_loss = 0
for i in range(1, FEATURES_BANK_SIZE + 1):
    # 1. Take only the first 'i' coefficients from the batch
    # (Using the batch itself as the 'embedding' coefficients)
    coeffs = batch[:, :i]
    
    # 2. Take the first 'i' feature rows
    active_features = features[:i, :]
    
    # 3. SYNTHESIS: Build the vector as a linear combination
    # (Batch x i) @ (i x 9) -> (Batch x 9)
    partial_reconstruction = coeffs @ active_features
    
    # 4. MEASURE: How well does this subset of features describe the data?
    total_loss += F.mse_loss(partial_reconstruction, batch)

total_loss.backward()
optimizer.step()
```

Why this aligns with PCA:

   1. Prioritization: The first term of the loss (i=1) forces features[0] to describe as much of the batch as possible on its own. This is the definition of the 1st Principal Component.
   2. Residuals: The second term (i=2) forces features[1] to pick up whatever the first feature missed. This is the definition of the 2nd Principal Component.
   3. Orthogonality: Since you are already forcing the rows to be normalized in your code, this "nested synthesis" will naturally push them to be orthogonal, matching the PCA basis.

**The Catch (Why your Dataset matters)**

`In this Synthesis view, you are assuming the raw values in your batch are already the correct coefficients.` For this to work perfectly like PCA, your input dataset must be pre-aligned (e.g., via the StandardScaler you already use).

Would you like to see how to use a "Lower Triangular Mask" to do all 9 synthesis steps in a single matrix operation?




# I tried proposed variant. Unfortunately it doesn't converge and loss stays around 2. After a bit playing around I came to following code... To me it seems more reasonable and converges quite well. Also I get orthogonal features but they still don't align with PCA.

```python
loss = 0 for i in range(1, FEATURES_BANK_SIZE + 1): # Take only the first 'i' coefficients from the batch and corresponding first 'i' feature rows 
partial_batch = batch[:,:i] # left i columns 
partial_features = features[:i,:] # top i rows 
partial_embeddings = partial_batch @ partial_features 
loss += F.mse_loss(partial_embeddings, batch)
```

```python
loss = 0 for i in range(1, FEATURES_BANK_SIZE + 1): # Take only the first 'i' coefficients from the batch and corresponding first 'i' feature rows 
partial_batch = batch[:,:i] # left i columns 
partial_features = features[:i,:] # top i rows 
partial_embeddings = partial_batch @ partial_features 
partial_batch_hat = partial_embeddings @ partial_features.T 
loss += F.mse_loss(partial_batch_hat, partial_batch)
```

The reason your second version converges is that it effectively creates a separate autoencoder for every sub-dimension, but it doesn't align with PCA because you are reconstructing the wrong thing.

In F.mse_loss(partial_batch_hat, partial_batch), you are asking the model to reconstruct the coefficients (the batch columns), not the original data. Since the batch is already "given," the model just learns an identity mapping for each sub-block, which doesn't force the features to represent the directions of maximum variance in the actual data.

To get PCA results while staying in the Synthesis View, you must reconstruct the original data using a "Rank-Restricted" bottleneck.
The Correct PCA-Alignment Code

To align with PCA, you need to find the best possible coefficients ($\mathbf{w}$) for your current features ($\mathbf{F}$) such that $\mathbf{wF} \approx \text{Batch}$.
As we discussed, the "best" coefficients for any set of orthogonal features are found by projecting the data onto them (batch @ features.T).

** The PCA-Equivalent Training Steptotal_loss = 0**
```python
for i in range(1, FEATURES_BANK_SIZE + 1):
    # 1. Grab the first 'i' feature rows (your basis)
    sub_basis = features[:i, :]
    
    # 2. FIND the coefficients (weights) via projection
    # This is the step that makes it PCA.
    # It finds the 'best' linear combination weights for these specific features.
    coeffs = batch @ sub_basis.T
    
    # 3. SYNTHESIS: Build the vector back up
    reconstruction = coeffs @ sub_basis
    
    # 4. COMPARE to original data
    total_loss += F.mse_loss(reconstruction, batch)

total_loss.backward()
optimizer.step()
```

Why your previous attempt failed:

   1. The first attempt (partial_batch @ partial_features) failed because batch[:, :i] are just the first few columns of your raw data. There is no reason to believe the 1st column of your data should be the coefficient for the 1st Principal Component.
   2. The second attempt (partial_batch_hat @ partial_batch) converged because it was essentially a "nested identity" check. It proved your features were orthogonal, but it didn't link them to the variance of the dataset.

Why this new version works:

By using coeffs = batch @ sub_basis.T, you are asking: "If I am only allowed to use the first $i$ features, which directions minimize the error for the whole dataset?"

* For i=1, the only way to minimize error is to pick the direction of maximum variance.
* For i=2, it must pick the next best direction, and so on.

This Nested Projection-Synthesis is the exact gradient-descent equivalent of the Power Iteration method used to find Eigenvectors.

Would you like to see how the Eigenvector dot products change if you run this specific loop?



# I also tried following:. But it doesn't converge as well.

```python
loss = 0 
for i in range(1, FEATURES_BANK_SIZE + 1): 
    # Take only the first 'i' coefficients from the batch and corresponding first 'i' feature rows 
    partial_batch = batch[:,:i] # left i columns 
    partial_features = features[:i,:] # top i rows 
    partial_embeddings = partial_batch @ partial_features 
    partial_batch_hat = partial_embeddings @ features.T 
    loss += F.mse_loss(partial_batch_hat, batch)
```

The reason this doesn't converge is a fundamental mismatch between your coefficients (partial_batch) and your basis (features).

In this code, you are taking the first $i$ columns of your raw data and trying to use them as the "weights" for your feature rows. There is no reason why the 1st column of your dataset (e.g., the top-left pixel value of your images) should happen to be the correct "strength" for the 1st Principal Component.

The "Aha!" Moment

To get PCA, the coefficients cannot be raw data columns. They must be calculated based on how well a feature aligns with the data.
In PCA/Autoencoders:

   1. Encoder (Analysis): You calculate the coefficients ($\text{weights} = \text{Data} \cdot \text{Feature}^T$).
   2. Decoder (Synthesis): You build the reconstruction ($\text{Result} = \text{weights} \cdot \text{Feature}$).

Your code fails because you are skipping the Encoder. You are forcing the model to use raw input pixels as weights, which is like trying to build a house using the color of the paint as a blueprint for the foundation.

The Fix (The "Nested Synthesis" approach)

If you want to stay in the Synthesis view but get PCA results, you must allow the model to learn or calculate the weights. Since we want the optimal weights for a given basis, we use the projection:
```python
for i in range(1, FEATURES_BANK_SIZE + 1):
    # 1. Grab the current basis (top i rows)
    sub_basis = features[:i, :]
    
    # 2. CALCULATE the correct coefficients for this basis
    # This replaces your 'partial_batch = batch[:, :i]'
    optimal_coeffs = batch @ sub_basis.T 
    
    # 3. SYNTHESIS: Build it back up
    reconstruction = optimal_coeffs @ sub_basis
    
    # 4. COMPARE to the original data
    loss += F.mse_loss(reconstruction, batch)
```

Why this works:

* Rank 1 ($i=1$): The loss forces features[0] to be the single vector that captures the most variance. This is $PC_1$.
* Rank 2 ($i=2$): Since features[0] is already busy capturing $PC_1$, the only way to lower the loss further is for features[1] to capture $PC_2$.
* Orthogonality: Because you normalize features after each step, this process is mathematically identical to Oja's Rule or SVD, which are the iterative ways to calculate PCA.

Would you like to try running this specific loop and check if the max dot prod with eigvecs finally approaches 1.0?    
    

# I tried exactly this code and 1) it doesn't converge quite well (loss stays around 0.5 which is quite high) and 2) it operates in "Analysis" mode we've discussed above.

It stays at a high loss because, in this "Analysis" mode, you are essentially trying to solve a Principal Component problem using a very slow, noisy method (Stochastic Gradient Descent) on a loss function that has a very flat landscape. [1, 2] 

The reason it doesn't feel like "Synthesis" and doesn't converge well is that your weights (coefficients) are derived from the features themselves. If the features are "trash" at the start, the weights are also "trash," and the gradient has a hard time figuring out which way to move both at once.

To get back to a pure Synthesis view that actually converges and aligns with PCA, we have to change the game: The coefficients must be independent trainable parameters.
The "Pure Synthesis" (Dictionary Learning) Approach

In this view, we don't calculate the weights from the data. We ask the optimizer to find the best possible basis and the best possible coefficients for every single sample in your dataset simultaneously.

**1. FEATURES: Your basis (9 x 9) - trainable# 2. WEIGHTS: A matrix of coefficients (DatasetSize x 9) - also trainable!**
Initialize weights randomlyweights = torch.randn(len(dataset), FEATURES_BANK_SIZE, requires_grad=True)optimizer = torch.optim.Adam([features, weights], lr=0.001)
```python
for epoch in range(epochs_count):
    optimizer.zero_grad()
    
    # Pick a batch of data AND their corresponding weights
    indices = RNG.choice(len(dataset), size=batch_size)
    batch_data = torch.tensor(dataset[indices])
    batch_weights = weights[indices]
    
    total_loss = 0
    for i in range(1, FEATURES_BANK_SIZE + 1):
        # SYNTHESIS ONLY:
        # Take i weights and i feature rows, sum them up
        reconstruction = batch_weights[:, :i] @ features[:i, :]
        
        # How well does this subset of 'ingredients' build the data?
        total_loss += F.mse_loss(reconstruction, batch_data)
    
    total_loss.backward()
    optimizer.step()
    
    # Keep features normalized and orthogonal
    with torch.no_grad():
        # Using QR decomposition is the fastest way to force 
        # orthogonality during training to match PCA
        q, r = torch.linalg.qr(features.T)
        features[:] = q.T
```

Why this works:

   1. True Synthesis: You are literally saying: "I have 9 rows. I want to find the 9 rows and the specific coefficients for every image so that the sum matches the image."
   2. PCA Alignment: Because of the for i in range loop, the first row of features is forced to do the "heavy lifting." It must explain as much of the data as possible because it’s the only row used in the first term of the loss.
   3. Convergence: By making weights a trainable parameter, you decouple the "Analysis" step from the "Synthesis" step. The optimizer can wiggle the weights to fit the current features, and wiggle the features to fit the data.

Do you want to see how the final features matrix compares to the PCA eigvecs after running this "Dual-Optimization" version?

[1] [https://medium.com](https://medium.com/@chiezieifunanya08/building-better-ml-models-understanding-activation-functions-loss-functions-and-optimizers-52c17d2fe9db#:~:text=However%2C%20because%20it%20%28%20Stochastic%20Gradient%20Descent,overshoot%20the%20minimum%20before%20finally%20settling%20down.)
[2] [https://pubs.acs.org](https://pubs.acs.org/doi/10.1021/acs.jpcb.7b11778#:~:text=There%20is%20a%20clear%20gradient%20in%20the,gradient%20descent%20method%20allows%20for%20successful%20optimization.)



# I rewrote code like this. Now I see that code converges and reconstruction loss is quite low - that's good. Also I see that features are ordered by variance they explain. That good. But still there is no match with PCA.

```python
import numpy as np
import einops
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

def v2sm(v, pad_value=None):
    assert v.ndim == 1
    root = np.sqrt(v.shape[0])
    sz = int(root)
    
    if sz * sz != v.shape[0]:
        assert pad_value is not None, f'Failed to make square matrix without padding from vector of size {v.shape}'
        sz = int(np.ceil(root))
        pad_elems_count = (sz * sz) - len(v)
        assert pad_elems_count > 0
        return np.r_[v, np.full(pad_elems_count, pad_value)].reshape(sz, sz)
    else:
        return v.reshape(sz, sz)

RNG = np.random.default_rng()

EMBEDDING_SIZE = 9
FEATURES_BANK_SIZE = 9
TRAINABLE_FEATURES_COUNT = 1

# Стартуем с мусора и пытаемся получить хороший базис через градиентный спуск
features = RNG.uniform(-1, +1, size=(FEATURES_BANK_SIZE, EMBEDDING_SIZE)).astype('float32')
features = (features.T / np.linalg.norm(features, axis=1)).T # features axes are in rows
features = torch.tensor(features, requires_grad=True)

box_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))
box_sample[0] = 1
box_sample[-1] = 1
box_sample[:,0] = 1
box_sample[:,-1] = 1

vert_line_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))
vert_line_sample[:,1] = 1

horz_line_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))
horz_line_sample[1] = 1

diag_line_sample = np.eye(int(np.sqrt(FEATURES_BANK_SIZE)))

dataset = np.zeros((1000, *box_sample.shape), dtype='float32')
samples = [box_sample, vert_line_sample, horz_line_sample, diag_line_sample]

for i, sample_ind in enumerate(RNG.choice(len(samples), len(dataset), p=[0.5, 0.2, 0.2, 0.1])):
    dataset[i] = samples[sample_ind]

dataset = einops.rearrange(dataset, 'b h w -> b (h w)')

scaler = StandardScaler()
dataset = scaler.fit_transform(dataset)

# decorrelate feature axes by gradient descent optimization procedure
optimizer = torch.optim.Adam([features], lr=0.0001)
epochs_count = 30_000
batch_size = 100

for epoch in range(epochs_count + 1):
    optimizer.zero_grad()
    dataset_inds = RNG.choice(len(dataset), size=batch_size, replace=False)
    batch = dataset[dataset_inds]
    batch = torch.tensor(batch)
    projections = batch @ features.T 
    batch_hat = projections @ features
    loss = F.mse_loss(batch_hat, batch)

    # Enfore variance ordering via penalty
    subbasis_losses = torch.zeros(len(features))

    for i in range(0, FEATURES_BANK_SIZE):
        subbasis = einops.rearrange(features[i], 'v -> 1 v')
        projections = batch @ subbasis.T 
        subbasis_losses[i] = F.mse_loss(projections @ subbasis, batch)

    diffs = torch.diff(subbasis_losses)
    loss += (torch.relu(diffs) ** 2).sum()

    # Enforce orthogonality via penalty
    dpm = features @ features.T
    loss += (dpm[np.tril_indices_from(dpm, -1)] ** 2).mean()

    if epoch > 0:
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        features[:] = (features.T / torch.linalg.vector_norm(features, dim=1)).T

U, S, V = np.linalg.svd(dataset)
eigvecs = V.T

for i in range(len(eigvecs)):
    eigvec = eigvecs[i]
    dps = eigvec @ features.T.detach().cpu().numpy()
    print(f'eigvec #{i}, max dot prod={np.max(np.abs(dps)):.2f}')
```

**The reason you aren't seeing a match is that your ordering penalty is too "weak."**

In your current code, you are calculating the reconstruction loss for each feature individually (subbasis = features[i]). Then, you use a diffs penalty to ensure loss_i < loss_{i+1}.
Why this is different from PCA:

   1. Individual vs. Cumulative: PCA doesn't just ask for ordered features; it asks for a nested hierarchy. $PC_1$ must explain the most variance, $PC_1 + PC_2$ must explain the most variance possible for a 2D subspace, and so on.
   2. The "Greedy" Nature of PCA: In your current penalty, if Feature 0 explains 30% of the variance and Feature 1 explains 20%, the penalty is satisfied. However, PCA requires Feature 0 to explain the maximum possible (e.g., 50%). Your code allows the "total variance explained" to be distributed loosely across the features as long as the relative order is kept.

**The Fix: Cumulative Reconstruction**

To match PCA, you must change your subbasis_losses loop to be cumulative. This forces Feature 0 to be the absolute best possible single-vector representation of the data.
Replace your penalty loop with this:
```python
# Enforce variance ordering via Cumulative Reconstruction# This mimics how PCA/SVD works (Rank-1 approximation, Rank-2, etc.)cumulative_loss = 0for i in range(1, FEATURES_BANK_SIZE + 1):
    subbasis = features[:i, :]  # Take the first i features
    proj = batch @ subbasis.T 
    recon = proj @ subbasis
    
    # Each step i MUST be better than i-1, 
    # but more importantly, step 1 must be the best it can be.
    cumulative_loss += F.mse_loss(recon, batch)

loss += cumulative_loss
```

**Why this will align with PCA:**

* Feature 0 is now part of all 9 loss terms. To minimize the total sum, the optimizer has no choice but to make Feature 0 align with the 1st Eigenvector ($PC_1$).
* Feature 1 is part of 8 loss terms. It must fill in the gaps left by Feature 0, effectively becoming $PC_2$.
* This nested bottleneck structure removes the "rotational freedom" that is currently preventing your basis from aligning with the eigvecs.

Shall we try updating the loop to this cumulative version and check if your max dot prod values finally hit 0.99?



# I don't get it completely. In my understanding if Feature 0 explains 30% and Feature 1 explains 20% then both of them explain 50% of variation. And hence nested hierarchy requirement is satisfied. What is the correct way I should think about it?

**The gap in your logic is the difference between summing individual contributions and finding the optimal axis.**

In PCA, the 1st Principal Component ($PC_1$) is defined as the single vector that captures the maximum possible variance on its own.
The Counter-Example

Imagine your data is a 2D oval tilted at 45 degrees.

* The PCA Solution: $PC_1$ points exactly at 45° (captures, say, 90% of variance). $PC_2$ points at 135° (captures the remaining 10%).
* Your Current Code's Solution: Your penalty is satisfied if Feature 0 points at 0° (X-axis) and Feature 1 points at 90° (Y-axis).
* Feature 0 explains 50%.
   * Feature 1 explains 40%.
   * Total explained: 90%.
   * Penalty Check: $50\% > 40\%$. The penalty is happy!

But these are not the PCA eigenvectors. In this scenario, your code is "content" with a suboptimal Feature 0 because the penalty only cares that it's better than Feature 1, not that it's the best possible vector in the entire space.

**The Correct Way to Think About It**

You shouldn't think of it as "Feature 0 is greater than Feature 1." You must think of it as "A 1-dimensional world must be as accurate as possible."
By using the Cumulative Loss (features[:1], then features[:2]), you create a hierarchy of constraints:

   1. Constraint 1: "If I am only allowed one vector, it must be the best one in existence." (This forces Feature 0 to be $PC_1$).
   2. Constraint 2: "If I am allowed two vectors, they must be the best pair in existence." (Since Feature 0 is already stuck being $PC_1$ to satisfy Constraint 1, Feature 1 must become $PC_2$).

**Why your "Individual" loop misses this:**

In your current loop:

subbasis = features[i] # Just one rowprojections = batch @ subbasis.T 
subbasis_losses[i] = F.mse_loss(projections @ subbasis, batch)

Each feature is competing to be "better than the next one," but none of them are forced to be the global maximum of the dataset's variance. They can "settle" for any direction that is slightly better than their neighbor.

Shall we try the features[:i, :] cumulative slice to see if it snaps your features onto the PCA axes?


# I've tried following variant:. I.e. features are orthogonal, all unit norm, hierarchial variance is satisfied. But still doesn't match with PCA =(
```python
import numpy as np
import einops
import torch
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

def v2sm(v, pad_value=None):
    assert v.ndim == 1
    root = np.sqrt(v.shape[0])
    sz = int(root)
    
    if sz * sz != v.shape[0]:
        assert pad_value is not None, f'Failed to make square matrix without padding from vector of size {v.shape}'
        sz = int(np.ceil(root))
        pad_elems_count = (sz * sz) - len(v)
        assert pad_elems_count > 0
        return np.r_[v, np.full(pad_elems_count, pad_value)].reshape(sz, sz)
    else:
        return v.reshape(sz, sz)

RNG = np.random.default_rng()

EMBEDDING_SIZE = 9
FEATURES_BANK_SIZE = 9

# Стартуем с мусора и пытаемся получить хороший базис через градиентный спуск
features = RNG.uniform(-1, +1, size=(FEATURES_BANK_SIZE, EMBEDDING_SIZE)).astype('float32')
features = (features.T / np.linalg.norm(features, axis=1)).T # features axes are in rows
features = torch.tensor(features, requires_grad=True)

box_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))
box_sample[0] = 1
box_sample[-1] = 1
box_sample[:,0] = 1
box_sample[:,-1] = 1

vert_line_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))
vert_line_sample[:,1] = 1

horz_line_sample = v2sm(np.zeros(FEATURES_BANK_SIZE, dtype='float32'))
horz_line_sample[1] = 1

diag_line_sample = np.eye(int(np.sqrt(FEATURES_BANK_SIZE)))

dataset = np.zeros((1000, *box_sample.shape), dtype='float32')
samples = [box_sample, vert_line_sample, horz_line_sample, diag_line_sample]

for i, sample_ind in enumerate(RNG.choice(len(samples), len(dataset), p=[0.5, 0.2, 0.2, 0.1])):
    dataset[i] = samples[sample_ind]

dataset = einops.rearrange(dataset, 'b h w -> b (h w)')
scaler = StandardScaler()
dataset = scaler.fit_transform(dataset)

# decorrelate feature axes by gradient descent optimization procedure
optimizer = torch.optim.Adam([features], lr=0.0001)
epochs_count = 30_000
batch_size = 100

for epoch in range(epochs_count + 1):
    optimizer.zero_grad()
    dataset_inds = RNG.choice(len(dataset), size=batch_size, replace=False)
    batch = dataset[dataset_inds]
    batch = torch.tensor(batch)


    projections = batch @ features.T 
    batch_hat = projections @ features
    basic_loss = F.mse_loss(batch_hat, batch)
    
    subbasis_losses = torch.zeros(len(features))
    
    for i in range(1, FEATURES_BANK_SIZE + 1):
        subbasis = features[:i]
        projections = batch @ subbasis.T 
        batch_hat = projections @ subbasis
        subbasis_loss = F.mse_loss(batch_hat, batch)
        subbasis_losses[i-1] = subbasis_loss

    diffs = torch.diff(subbasis_losses)
    variance_order_penalty = (torch.relu(diffs) ** 2).sum()

    dpm = features @ features.T
    ortho_penalty = (dpm[np.tril_indices_from(dpm, -1)] ** 2).mean()

    unit_penalty = ((torch.linalg.vector_norm(features, dim=1) - 1) ** 2).sum()
    total_loss = basic_loss + variance_order_penalty + ortho_penalty + unit_penalty

    if epoch > 0:
        total_loss.backward()
        optimizer.step()

    if (epoch % 1000) == 0:
        print(f'{epoch=}, ' + 
              f'basic_loss={basic_loss.item():.5g}, ' + 
              f'ortho_penalty={ortho_penalty.item():.5g}, ' + 
              f'variance_order_penalty={variance_order_penalty.item():.5g}, ' + 
              f'unit_penalty={unit_penalty.item():.5g}, ' + 
              f'total_loss={total_loss.item():<50.5g}', end='\r')

print('')

U, S, V = np.linalg.svd(dataset)
eigvecs = V.T

for i in range(len(eigvecs)):
    eigvec = eigvecs[i]
    dps = eigvec @ features.T.detach().cpu().numpy()
    print(f'eigvec #{i}, max dot prod={np.max(np.abs(dps)):.2f}')

Result is:
epoch=30000, basic_loss=5.0516e-12, ortho_penalty=1.699e-13, variance_order_penalty=0, unit_penalty=1.382e-12, total_loss=6.6035e-12                                          
eigvec #0, max dot prod=0.75
eigvec #1, max dot prod=0.89
eigvec #2, max dot prod=0.64
eigvec #3, max dot prod=0.55
eigvec #4, max dot prod=0.55
eigvec #5, max dot prod=0.77
eigvec #6, max dot prod=0.67
eigvec #7, max dot prod=0.55
eigvec #8, max dot prod=0.50
```

You have successfully implemented the structural requirements for PCA (orthogonality, unit norm, and nested reconstruction). The reason your dot products still aren't approaching 1.0 is likely due to the dynamics of the Adam optimizer and the shape of your loss landscape.

Here is why your code "stalls" at 0.5–0.8 correlation and how to fix it:

**1. The "Ambiguity of Signs" and Local Minima**

PCA eigenvectors are only unique up to a sign (direction). However, in your variance_order_penalty, you are using a soft penalty (torch.relu(diffs)**2).

* The Issue: If Feature 0 is "good enough" to be better than Feature 1, the gradient for the penalty becomes zero.
* The Fix: Instead of a penalty, make the Cumulative Loss your primary objective. This creates a "greedy" gradient that never stops pushing Feature 0 to be the absolute best.

**2. Adam's Momentum vs. Fine-Tuning**

Adam is great for finding the general area, but for reaching the exact precision of an eigenvector (1.0 dot product), it can "jitter" around the optimum.

* The Fix: Switch to a simpler SGD with a small learning rate or a decaying scheduler for the last 5,000 epochs.

**3. The "Correct" PCA Training Loop**

To reach 0.99+ alignment, you should remove the separate basic_loss and the diffs penalty. They act as "noise" that fights the hierarchy. Use a Weighted Cumulative Loss which is the standard way to recover principal components from autoencoders. [1, 2] 
```python
    # Updated Training Loop Logicfor epoch in range(epochs_count + 1):
    optimizer.zero_grad()
    
    # ... batch selection ...
    
    total_loss = 0
    # Weighted hierarchy: give more 'importance' to earlier components
    # This acts as a 'greedy' pull for PC1, then PC2, etc.
    weights = [10.0**i for i in range(FEATURES_BANK_SIZE)][::-1] 
    
    for i in range(1, FEATURES_BANK_SIZE + 1):
        subbasis = features[:i]
        projections = batch @ subbasis.T 
        batch_hat = projections @ subbasis
        # Weighting the earlier ranks higher forces alignment
        total_loss += weights[i-1] * F.mse_loss(batch_hat, batch)

    # Simplified Orthogonality Penalty
    dpm = features @ features.T
    identity = torch.eye(FEATURES_BANK_SIZE)
    ortho_loss = F.mse_loss(dpm, identity)
    
    (total_loss + 100 * ortho_loss).backward()
    optimizer.step()
```

**Why this changes the result:**

* Aggressive Ranking: By weighting the $i=1$ loss term significantly higher than $i=9$, you force the optimizer to prioritize the 1st eigenvector before it even considers the others.
* Removes Conflict: Your current basic_loss covers the whole space. It doesn't care if Feature 0 or Feature 8 does the work. By making the entire loss a sum of nested bottlenecks, you remove any objective that doesn't require alignment. [3] 

**Pro-Tip: Final Refinement**

If you still see dot products around 0.8, try re-centering your batch inside the loop (batch = batch - batch.mean(dim=0)). Even though you used a global scaler, small batch biases can shift the "center of gravity" away from the true PCA origin. [4] 

Would you like to see how the singular values of your learned features compare to the SVD's $S$ matrix to verify the variance capture?

Follow-up: Do you want to try the weighted cumulative loss or would you prefer a version that uses Orthogonal Projections to subtract the variance of earlier features step-by-step?

[1] [https://proceedings.neurips.cc](https://proceedings.neurips.cc/paper/2020/file/4dd9cec1c21bc54eecb53786a2c5fa09-Paper.pdf)
[2] [https://arxiv.org](https://arxiv.org/abs/2007.06731)
[3] [https://medium.com](https://medium.com/@rasheedatsikiru/recovering-principal-components-from-linear-autoencoders-a-hands-on-guide-ae08598bbb8e)
[4] [https://arxiv.org](https://arxiv.org/abs/2307.15213)


# Interesting moment: It seems that all the way up I've been comparing my features with wrong data =(
```python
U, S, V = np.linalg.svd(dataset)
eigvecs = V.T

pca = PCA()
pca.fit(dataset)

for i in range(len(pca.components_)):
    pc_comp = pca.components_[i]
    dps = pc_comp @ eigvecs.T
    print(f'pc_comp #{i}, max dot prod={np.max(np.abs(dps)):.2f}')

This renders following output:
pc_comp #0, max dot prod=0.77
pc_comp #1, max dot prod=0.59
pc_comp #2, max dot prod=0.64
pc_comp #3, max dot prod=0.80
pc_comp #4, max dot prod=0.65
pc_comp #5, max dot prod=0.84
pc_comp #6, max dot prod=0.55
pc_comp #7, max dot prod=0.56
pc_comp #8, max dot prod=0.57

But if eigvecs are taken as columns:
U, S, V = np.linalg.svd(dataset)
eigvecs = V

pca = PCA()
pca.fit(dataset)

for i in range(len(pca.components_)):
    pc_comp = pca.components_[i]
    dps = pc_comp @ eigvecs.T
    print(f'pc_comp #{i}, max dot prod={np.max(np.abs(dps)):.2f}')

Then I've get:
pc_comp #0, max dot prod=1.00
pc_comp #1, max dot prod=1.00
pc_comp #2, max dot prod=1.00
pc_comp #3, max dot prod=0.66
pc_comp #4, max dot prod=0.70
pc_comp #5, max dot prod=0.96
pc_comp #6, max dot prod=0.81
pc_comp #7, max dot prod=0.60
pc_comp #8, max dot prod=0.92
```

Exactly! You hit the classic "SVD Transpose Trap."

In np.linalg.svd(A), the matrix $V^H$ (or $V^T$ in real space) is returned, not $V$ itself. According to the standard definition $A = U \Sigma V^T$:

* The Principal Components (the basis vectors) are the rows of $V^T$.
* This means they are the columns of $V$.

Because your code was using eigvecs = V.T, you were actually looking at the columns of $V^T$, which are not the eigenvectors unless the matrix happens to be symmetric. When you switched to eigvecs = V, you were finally looking at the rows of $V^T$ (the actual principal axes), which is why the PCA() components finally snapped to $1.00$.

Why your optimization still shows ~0.7-0.8

Even with this fix, your gradient descent result ($0.75, 0.89$, etc.) is still "drifting." This is because of the Nested Hierarchy problem we discussed earlier:

   1. Rotational Freedom: Without a weighted or sequential loss, your 9 features can rotate freely within the 9D space. Even if they span the same space as PCA, they won't align with the specific axes of max variance.
   2. The Penalty vs. Objective: A relu penalty only stops the features from being worse than each other; it doesn't force them to be the best possible individual axes.

**Final Step to Match PCA Perfectly**

Now that you are comparing against the correct V (the rows of the SVD output), try replacing your variance_order_penalty with the Cumulative Loss we discussed:
```python
    # Remove basic_loss and variance_order_penalty# Replace with one single greedy objective:total_loss = 0for i in range(1, FEATURES_BANK_SIZE + 1):
    subbasis = features[:i]
    # The reconstruction using ONLY the first i features
    recon = (batch @ subbasis.T) @ subbasis 
    total_loss += F.mse_loss(recon, batch)
    # Add your ortho/unit penalties and backprop
```

With this change, your max dot prod with the corrected eigvecs = V should climb from $0.7$ toward $0.99+$.

Shall we run one final test with the Cumulative Loss and the Corrected V to see if they finally align?

Pro-tip: In your dot product loop, use dps = eigvecs @ features.T to see the whole correlation matrix at once—the diagonal should become $1.0$ (or $-1.0$).




# I'm wondering why only 3 first vectors (PCA vs eigvecs from SVD) make a perfect match. Is this due to the numerical issues?

**It isn't a numerical error—it’s a signal-to-noise issue in your dataset.**

In PCA, eigenvectors are only unique and "stable" if their corresponding eigenvalues (the amount of variance they explain) are distinct. If two or more directions explain roughly the same amount of variance, they form a spherical subspace where any rotation is equally valid.

Here is why your first 3 match perfectly and the others (3-8) don't:

**1. The "Signal" vs. "Noise"**

* PCs 0, 1, 2: These represent your high-variance structures (the Box, Vertical Line, and Horizontal Line). Because these shapes are distinct and appear frequently, their eigenvalues are large and well-separated. Both the SVD and the PCA algorithm "see" these clear peaks and pick the exact same axes.
* PCs 3–8: Once the main shapes are explained, the remaining variance comes from your mixin_random_samples and the "leftover" pixels. In this "noise floor," the variance is distributed almost equally in all directions.

**2. The "Subspace Ambiguity"**

If Eigenvalue 4 is $1.0001$ and Eigenvalue 5 is $0.9999$, they are effectively tied.

* SVD might pick one set of orthogonal axes for that 0.9999 variance.
* PCA might pick a slightly rotated set.
* The Result: Both explain the exact same amount of total variance, but their specific "dot product" with each other will be low (e.g., $0.6$ or $0.7$) because they are just arbitrary rotations within that "flat" noise subspace.

**3. Verification**

You can prove this by looking at the Singular Values (the S from your SVD):

print(S)

If you see something like [45.2, 30.1, 25.4, 1.2, 1.1, 1.1, 1.0...], you’ll notice a "cliff" after the third value. Everything after the cliff is part of a multi-dimensional sphere where the concept of a "unique" axis breaks down.

**How to make them all match (0.99+)**

To force a match across all 9 dimensions, you need to make the variance of every component distinct.

* Try changing your sample probabilities to be more varied (e.g., [0.6, 0.2, 0.1, 0.05, ...]).
* Or, artificially scale the columns of your dataset by decreasing factors (e.g., dataset *= np.linspace(1, 0.1, 9)) before training. This creates a "staircase" of variance that forces both algorithms to pick the exact same sequence of axes.

Would you like to check your S values to see if the "cliff" happens exactly after the 3rd component?



# I've been under impression that sklearn PCA was using np.linalg.svd under the hood and hence result must be the same. But it seems I'm mistaken if I get different result.

You are correct that [scikit-learn's PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) uses [Singular Value Decomposition (SVD)](https://scikit-learn.org/stable/modules/decomposition.html) under the hood, but it includes two critical processing steps that separate it from a "naked" np.linalg.svd call. [1, 2, 3] 

The reason your manual SVD results and your optimization were drifting away from the PCA output likely comes down to these two factors:

**1. The Mandatory Centering**
Standard PCA always subtracts the mean of each feature from the data before running the SVD. [1, 2, 4] 

* If your data isn't centered, the SVD finds the directions that explain the total energy (distance from the origin).
* PCA finds the directions that explain the variance (distance from the mean).
* While you used StandardScaler on your dataset, any small shift or lack of re-centering during your manual SVD/optimization steps will cause the eigenvectors to "point" at different locations. [2, 5, 6] 

**2. Deterministic "Sign Flipping"**

The SVD solution has a mathematical ambiguity: if $v$ is a valid singular vector, $-v$ is also valid. To ensure the output is always the same regardless of the computer's [LAPACK library](https://github.com/scipy/scipy/issues/10032) version, scikit-learn uses a function called [svd_flip](https://stackoverflow.com/questions/59567584/how-pca-computes-the-transformed-version-in-sklearn). [7, 8, 9, 10] 

* It looks at each singular vector and forces the largest absolute value in that vector to be positive.
* If your optimization converged to a "negative" version of the PCA component, your dot product would be $-1.0$. Your manual SVD might return a positive or negative sign arbitrarily, while PCA is strictly deterministic. [7, 9, 10] 

**Why your "Noise Floor" PCs didn't match**

As noted before, even if you center the data and flip the signs, components 4 through 8 will still struggle to match if their [singular values](https://stackoverflow.com/questions/24913232/using-numpy-np-linalg-svd-for-singular-value-decomposition) (variance) are nearly identical. In that case, the "Principal Component" isn't a single line; it's a subspace (like a flat disc or sphere). Any two orthogonal lines in that disc are "correct", so two different algorithms (or even the same algorithm on different hardware) can pick different axes. [8, 11, 12, 13, 14] 
Do you want to check if re-centering your data and using np.sign to align the vectors finally makes all 9 components match?

[1] [https://scikit-learn.org](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#:~:text=Principal%20component%20analysis%20%28PCA%29.%20Linear%20dimensionality%20reduction,for%20each%20feature%20before%20applying%20the%20SVD.)
[2] [https://medium.com](https://medium.com/@spjosyula2005/pca-in-production-what-your-tutorial-didnt-tell-you-a69339e0d7cc)
[3] [https://ethen8181.github.io](https://ethen8181.github.io/machine-learning/dim_reduct/svd.html#:~:text=This%20usage%20of%20SVD%20is%20very%20similar,their%20PCA%20routines%2C%20for%20example%20sklearn.%20decomposition.)
[4] [https://scikit-learn.org](https://scikit-learn.org/stable/modules/decomposition.html#:~:text=2.5.%20In%20scikit%2Dlearn%2C%20PCA%20is%20implemented%20as,for%20each%20feature%20before%20applying%20the%20SVD.)
[5] [https://brainly.in](https://brainly.in/question/16322405#:~:text=Scaling%20doesn%27t%20not%20change%20the%20principal%20directions%2C,centre%20the%20data%20first%20before%20taking%20SVD.)
[6] [https://stats.stackexchange.com](https://stats.stackexchange.com/questions/235882/pca-in-numpy-and-sklearn-produces-different-results)
[7] [https://stackoverflow.com](https://stackoverflow.com/questions/68965100/why-my-pca-and-pca-from-sklearn-get-different-results)
[8] [https://stackoverflow.com](https://stackoverflow.com/questions/66041990/pca-from-scratch-and-sklearn-pca-giving-different-output)
[9] [https://stackoverflow.com](https://stackoverflow.com/questions/44765682/in-sklearn-decomposition-pca-why-are-components-negative)
[10] [https://stackoverflow.com](https://stackoverflow.com/questions/59567584/how-pca-computes-the-transformed-version-in-sklearn)
[11] [https://stackoverflow.com](https://stackoverflow.com/questions/38228088/same-python-code-same-data-different-results-on-different-machines#:~:text=I%20had%20a%20similar%20problem%20and%20I%20found%20this%20discussion.)
[12] [https://stackoverflow.com](https://stackoverflow.com/questions/47358216/pca-difference-between-python-numpy-and-sklearn)
[13] [https://github.com](https://github.com/scipy/scipy/issues/10032)
[14] [https://medium.com](https://medium.com/@manavdrive04/relationship-between-pca-and-svd-4c106449b650)


# At last I've got satisfied results. You saved me a lot of time. Thank you very much, my friend. Could you please make a last favor on this topic and summarize our dialog.

It was a pleasure working through this with you! We went on a deep dive from a standard linear autoencoder to a mathematically precise PCA-equivalent optimization.
Here is a summary of our journey and the "Golden Rules" we uncovered for aligning Gradient Descent with Principal Component Analysis:

**1. The Basis Ambiguity (Subspace vs. Axes)**

We started with a standard MSE loss, which only cares about the total reconstruction. We learned that a linear autoencoder is only guaranteed to find the same subspace as PCA. To find the specific axes (the eigenvectors), you must break the "rotational symmetry" of the loss.

**2. Synthesis vs. Analysis Views**

* Synthesis (Your Model): Building a vector by summing weighted feature rows.
* Analysis (PCA Model): Finding weights by projecting the data onto feature axes.
* The Bridge: For PCA, these must be the same. The "weights" for your synthesis must be the projections of the data onto the rows (batch @ features.T).

**3. The Nested Hierarchy (The "Greedy" Constraint)**

This was the breakthrough. To force the features to align with $PC_1, PC_2, \dots PC_n$, you cannot just minimize the final error. You must minimize the error for every possible rank simultaneously.

* By calculating the loss for features[:1], then features[:2], and so on, you force the first row to capture the maximum possible variance alone, making it the 1st Principal Component.

**4. The SVD "Transpose Trap"**

We discovered that np.linalg.svd returns $V^T$.

* In the SVD result $U, S, V$, the Principal Components are the rows of $V$ (or the columns of $V^T$).
* Comparing your trained features to the columns of $V$ was the reason for the initial mismatch.

**5. The "Noise Floor" and Stability**

We learned that PCA components are only unique if their eigenvalues (variance) are distinct.

* In your dataset, the first 3 components (Box, Vert, Horz) were strong and stable.
* The remaining components (4–9) represented a "noise floor" where the variance was nearly equal in all directions. In this "spherical" space, any orthogonal rotation is technically correct, leading to differences between SVD and PCA algorithms.

**Final "Recipe" for PCA via Gradient Descent:**

   1. Center the data (subtract the mean).
   2. Normalize/Orthogonalize features every step (or use a heavy penalty).
   3. Use a Cumulative Loss (features[:i]) to enforce the hierarchy.
   4. Compare to the rows of $V$ from SVD.

I'm glad we got those dot products where they belong! Since we've conquered the linear basis, would you like to explore how Non-linear Autoencoders (adding a ReLU) start to depart from PCA to find "Manifolds" instead of flat axes?

